<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Data to Decisions: GPU-Accelerated Decision Optimization</b></h1>
<h2><b>Exercise 3:</b> Accelerated LP Solver with cuOpt Python API</h2>
<br>

In this exercise, you will continue your exploration with cuOpt's algebraic modeling API, this time for a related linear programming (LP) problem.

> **Student challenge version:** Replace every `<<<TO DO>>>` marker with working Python code, then run the cells in order. The completed reference notebook with the same filename lives one folder up.

> Tip: if a cell raises a syntax error, look for the next `<<<TO DO>>>` marker in that cell.

<hr>

# **Linear Programming**

As briefly covered in the previous exercise, Linear Programming models also aim to help solve resource allocation problems under constraints by solving for decision variables `x_1, x_2, ...` appearing in the objective function and constraints of linear form. What makes LP models different from MILP models is the fact that all decision variables in the model are continuous.

LP models arise in diverse settings and industrial contexts such as product mix decisions, transportation problems, budget allocation problems and more, where decision variables can be allowed to have fractional values. Existing of well established techniques for solving LPs, such as the [**simplex method**](https://), [**interior point method**](https://), [**dual-simplex method**](https://), have made them more appealing than their mixed-integer counterparts due the solution complexity and combinatorial nature of the latter.

LP models also arise when a MILP model is too complex or takes too long to solve, and "relaxing" the integrality requirement of variables is acceptable. This is called the **"LP relaxation"** of a MILP model and depending on the goal of the optimization, it provides a lower bound (for minimization problems) or upper bound (for maximization problems), which helps understand the problem characteristics better.

cuOpt has two GPU-accelerated LP solvers, one known as **PDLP** and the other as **Barrier** (or Interior Point Method), that together solve many if not all LP problems faster than the optimization techniques mentioned above. The PDLP technique in particular, though still an emerging technique, is based on a first-order-method seeking to improve primal and dual solutions iteratively, while accelerating internal matrix operations on the GPU.

For the exercise in this notebook, we will revisit the supply chain production system we introduced and use cuOpt API to convert the MILP model into an LP model.

## Problem Definition

We will work on the same problem from the previous exercise, but relax the binary integer requirements of vendor-plant allocation variables.

Let's start by loading the model from the version we saved in the previous exercise. You will remember that a vendor can serve only one plant, as indicated by the value of the binary variable. We will relax this requirement so that a vendor can split its supply amongst multiple vendors.

The rest of the model remains more or less the same.

In [ ]:
# Load the previous problem saved as data/MIP_model.mps
from time import perf_counter
from cuopt.linear_programming.problem import Problem

# TODO 03.1: Load the model from Notebook 02.
problem = <<<TO DO>>>

# The production system has 6 vendors
Vendors = ['VH1', 'VH2', 'VH3', 'VH4', 'VL1', 'VL2']
Vendors_H = ['VH1', 'VH2', 'VH3', 'VH4']
Vendors_L = ['VL1', 'VL2']
Plants = ['P1', 'P2', 'P3']
Q = {'VH1': 250, 'VH2': 300, 'VH3': 400, 'VH4': 500, 'VL1': 450, 'VL2': 550}
D = {'P1': 350, 'P2': 450, 'P3': 700}
C = {
    'VH1': {'P1': 5, 'P2': 8, 'P3': 12},
    'VH2': {'P1': 7, 'P2': 6, 'P3': 9},
    'VH3': {'P1': 10, 'P2': 9, 'P3': 8},
    'VH4': {'P1': 11, 'P2': 10, 'P3': 7},
    'VL1': {'P1': 3, 'P2': 4, 'P3': 6},
    'VL2': {'P1': 4, 'P2': 3, 'P3': 5},
}


Visualizing this system with networkX should give us the exact same view we had in the previous exercise.

In [ ]:
# Function to plot the supply-chain network

import networkx as nx
import matplotlib.pyplot as plt

def draw_network(V, P, Q, D, C, X=None, Y=None):

    V = V[::-1]
    P = P[::-1]
    # Create bipartite graph
    B = nx.DiGraph()

    # Add nodes
    B.add_nodes_from(V, bipartite=0)
    B.add_nodes_from(P, bipartite=1)

    # Add edges from C
    for src, targets in C.items():
        for dst, weight in targets.items():
            if Y is None or (Y is not None and round(Y[src][dst].getValue(),2)!=0):
                B.add_edge(src, dst, weight=weight)

    # Layout for bipartite graph
    pos = nx.bipartite_layout(B, V)

    # Node labels
    labels_left = Q
    labels_right = D
    labels = {**labels_left, **labels_right}

    # Draw nodes and edges
    plt.figure(figsize=(7, 5))
    nx.draw(
        B, pos,
        with_labels=True,
        labels={node: f"{node}\n({labels[node]} lbs)" for node in B.nodes()},
        node_color=["skyblue" if n in V else "lightgreen" for n in B.nodes()],
        node_size=1500,
        font_size=8,
        edge_color="gray"
    )

    # Draw edge labels
    if X is None:
        edge_labels = {(u, v): f"${d['weight']}/lb" for u, v, d in B.edges(data=True)}
    else:
        edge_labels = {(u, v): f"{round(X[u][v].getValue(),2)} lbs\n${round(d['weight'] * X[u][v].getValue(), 2)}" for u, v, d in B.edges(data=True)}
    nx.draw_networkx_edge_labels(B, pos, edge_labels=edge_labels, font_size=8)

    plt.axis("off")
    plt.show()

draw_network(Vendors, Plants, Q, D, C)

<hr>

## **Step 1:** Modifying the MILP Model into an LP Model

Let's first check and see if the loaded model still has the decision variables as we originally designed: `CONTINUOUS` variables for amount of shipments between vendors and plants, and `INTEGER` variables for assigning vendors to plants.

In [ ]:
original_variables = problem.getVariables()
for v in original_variables:
    print(f" - Variable {v.VariableName} has type {v.VariableType}")

As expected, some variables are type C (CONTINUOUS) and the rest are type I (INTEGER).

Now let's change the assignment variables from `INTEGER` to `CONTINUOUS`.

In [ ]:
# Relax the MIP model to an LP model
# TODO 03.2: Create the LP relaxation.
lp_problem = <<<TO DO>>>
variables = lp_problem.getVariables()

for v_old, v in zip(original_variables, variables):
    var, vendor, plant = v.VariableName.split("_")
    (X if var == "x" else Y)[vendor][plant] = v
    print(
        f" - Variable [{v.VariableName}] relaxed from"
        f" {v_old.VariableType} in [{v_old.getLowerBound()}, {v_old.getUpperBound()}]"
        f" -> {v.VariableType} in [{v.getLowerBound()}, {v.getUpperBound()}]"
    )


When you make this modification, note that you still have the lower and upper bounds set as 0 and 1 for the ***previously binary*** variables. This means the assignment between a vendor and plant is now allowed to be a fractional value between 0 and 1.

#### **Sanity Checking Constraints**

It's a good idea now to check if our constraints are still meaningful and whether we need to revise or adjust them in the presence of this new reality.

Since you cannot allocate more than 100% of a vendor's supply, the sum of these variables for a vendor $v$ over all plants must not exceed 1, which means the very first constraint we added earlier to our model is still valid.

**The 2nd and 4th constraints are also still valid:**

2) Sum of all cheese-scrap pounds shipped from vendors to plant $p$ should be equal to the demand $D$ at plant $p$

```python
for P in Plants:
    problem.addConstraint(sum(X[V][P] for V in Vendors) == D[P])
```

4) Max Percentage of Low-Quality cheese should be 20%.

```python
u = 0.2
for P in Plants:
    problem.addConstraint((1-u) * sum(X[V][P] for V in Vendors_L) <= u * sum(X[V][P] for V in Vendors_H))
```

**The 3rd constraint may need a bit of attention:**

3) Pounds supplied by vendor $v$ ($X_{v,p}$) is less than or equal to the capacity $Q$ at the vendor

```python
for V in Vendors:
    for P in Plants:
        problem.addConstraint(X[V][P] <= Q[V]*Y[V][P])
```

Notice that the originally-binary variable $Y_{v,p}$ appears in this constraint on the right hand side. This used to force vendor $v$ to supply *nothing* to plant $p$ when the corresponding $Y$ variable $Y_{v,p}$ is set to 0. If $Y$ variables are now allowed to be fractional between 0 and 1, is this constraint still valid?

<details>
<summary><b>Answer</b></summary>

The constraint is still valid because when a fractional value (between 0 and 1) is multiplied with the vendor's available capacity `Q_v`, the result is the maximum amount that can be supplied by this vendor `v` to the respective plant `p`, and the `X_v,p` variable value will not be allowed to exceed that.

You may also question if this constraint is necessary in its current form. Why not have a different constraint instead in the form of

```
for V in Vendors:
    problem.addConstraint(sum(X[V][P] for P in Plants) <= Q[V])
```

which says the total amount shipped from vendor `v` to all plants is less than `Q` for that vendor. It turns out this would be another valid way to model this constraint, which also helps reduce the number of constraints in the model.

Our objective for the model is unchanged. We still want to minimize the total cost of shipment between vendors and plants.

In [ ]:
print(f"Objective of the problem is to {lp_problem.ObjSense} the Linear Expression:\n  0.0")
for v, c in lp_problem.getObjective().zipVarCoefficients():
    if v.getVariableName().startswith("x"):
        print(f"+ {c} {v.getVariableName()}")

<hr>

## **Step 2:** Solve the LP Model

We have successfully revised our problem in cuOpt. Let's solve it and take a look at how the solution is affected.

In [ ]:
# TODO 03.3: Time the LP solve and save the relaxed model.
start = perf_counter()
lp_problem.<<<TO DO>>>
elapsed = perf_counter() - start
print(f"Small LP relaxation solve time: {elapsed:.3f} seconds")

lp_problem.<<<TO DO>>>


With our model optimized to minimize the production system's costs, let us take a look at the details of the solution.

In [ ]:
print("|==================================================================|")
print("|                        Solution Metadata                         |")
print("|==================================================================|")
solve_time = lp_problem.SolveTime
primal_objective_value = lp_problem.ObjValue
print(f" Optimal Solution found in {solve_time:.3f} seconds")
print(f" Minimized cost of operating the production system is {primal_objective_value:.2f}\n")

print("|==================================================================|")
print("|                    Optimized Variable Values                     |")
print("|==================================================================|")
for V in Vendors:
    for P in Plants:
        x = X[V][P].Value
        y = Y[V][P].Value
        if round(y, 2) != 0:
            print(f" Vendor {V} ships {x:5.2f} lbs of cheese to Plant {P} ")

Notice how vendors `VH1` and `VL1` are now shipping to multiple plants `P1` and `P2`, thanks to the relaxation on the binary decision variables.

### **Did you also notice that the total cost of shipment is now less?**

It used to be `414` when we had the extra restriction that a vendor can only ship to one vendor, but now it's `394`. Removing this restriction allowed solver to explore other variations of shipping cheese in the supply chain, which resulted in reduced total cost.

This is actually what we meant by saying **an LP relaxation provides a lower bound** on the objective value for a minimization problem. The total cost will never go below `394` if you add other constraints to the model or add back the binary requirement for `Y` variables. This is why a MILP model relies a lot on its LP relaxation to achieve bounds that can help prune branches of the B&B tree.

Lastly, let us visualize the solution again ...

In [ ]:
draw_network(Vendors, Plants, Q, D, C, X, Y)

Here also you can see 2 lines emanating from `VH1` and `VL1` confirming that these two vendors ship to multiple plants.

## **Optimizing Larger Problems**

One of the reasons for solving an LP relaxation of a MILP model is to achieve a speedup over the MILP version that is more challenging due to its combinatorial nature. In the above example, maybe you have not observed this speedup since the problem is too small and it solves very fast.

Let's try the larger problem from the previous exercise and solve it as an LP. 
It loads as a MILP problem because some variables are integer-constrained, but we can run its LP relaxation by invoking the `relax()` function on the problem object.

## *Step 1: Read the MPS file and initialize the problem*

In [ ]:
# TODO 03.4: Load and solve the larger MILP for comparison.
big_problem = <<<TO DO>>>
print(f"""The supply-chain problem has:
 > {big_problem.NumVariables} variables
 > {big_problem.NumConstraints} constraints
 > {big_problem.NumNZs} non-zeros""")

start = perf_counter()
big_problem.<<<TO DO>>>
elapsed = perf_counter() - start

print(f"\nObjective Value of original solution is {big_problem.ObjValue}")
print(f"Original MILP solve time: {elapsed:.3f} seconds")


## *Step 2: Relax The Problem*

Relaxing the MIP problem is the equivalent of setting all the variables to CONTINUOUS and solving for that.

In [ ]:
# Manually create the LP relaxation of the larger MILP from a fresh model copy.
# TODO 03.5: Start from a fresh copy of the large model.
big_lp_problem = <<<TO DO>>>

# TODO 03.6: Relax each variable to continuous.
for v in big_lp_problem.getVariables():
    v.<<<TO DO>>>

# TODO 03.7: Refresh the model after changing variable attributes.
big_lp_problem.<<<TO DO>>>


## *Step 3: Solve The Model*

In [ ]:
# TODO 03.8: Time and solve the larger LP relaxation.
start = perf_counter()
big_lp_problem.<<<TO DO>>>
elapsed = perf_counter() - start

print(f"\nObjective Value of optimized solution is {big_lp_problem.ObjValue}")
print(f"Large LP relaxation solve time: {elapsed:.3f} seconds")


Note that the relaxed problem solves much faster than the original MILP in this example. 
The exact timing may vary by GPU and cuOpt version, but the important idea is that relaxing integer variables turns the model into a continuous LP, which is usually easier to solve. 
The optimal objective value for the relaxed model should be around `800,413.9`, compared with about `1,227,128` for the MILP version.

## **Alternative Solution**

The cuOpt library has a one line command that can be used instead. Complete this #TODO# with the right command. - use the cuopt documentation to find it. (Note, if the kernel crashes, comment the previous block and rerun the full notebook again.)

In [ ]:
# The one-line cuOpt helper creates the same LP relaxation.
# TODO 03.9: Use the one-line cuOpt relaxation helper.
big_lp_problem_alt = <<<TO DO>>>
big_lp_problem_alt.solve()

print(f"LP relaxation objective from manual variable update: {big_lp_problem.ObjValue:.1f}")
print(f"LP relaxation objective from relax(): {big_lp_problem_alt.ObjValue:.1f}")


Both approaches solve the same LP relaxation. 
The LP relaxation has a lower objective value than the original MILP because it removes the integer restrictions from the decision variables. 
For a minimization problem, that relaxed objective acts as a lower bound: it shows what would be possible if fractional production or assignment choices were allowed.

Let's take a peek at the solution from solving this LP version of the model:

In [ ]:
print("|==================================================================|")
print("|                        Solution Metadata                         |")
print("|==================================================================|")
solve_time = big_lp_problem.SolveTime
primal_objective_value = big_lp_problem.ObjValue
print(f" Optimal Solution found in {solve_time:.3f} seconds")
print(f" Minimized cost of operating the production system is {primal_objective_value:.2f}\n")

print("|==================================================================|")
print("|                    Optimized Variable Values                     |")
print("|==================================================================|")
value_dict = dict()
for v in big_lp_problem.getVariables():
    approx_value = round(v.Value, 7)  ## <- TODO: Try increasing rounding place
    value_dict[approx_value] = value_dict.get(approx_value, []) + [v.VariableName]

for value, keys in value_dict.items():
    print(f" * {len(keys):3.0f} variables have value close to {value}")
    if (preview := 0):                ## <- TODO: set preview := 10?
        print(" >  ", "|".join(keys[:preview]), "..." if len(keys)>preview else "", "\n")


In [ ]:
# TODO 03.10: Save the larger LP relaxation as data/cheese_lp.mps for later reuse.
big_lp_problem.<<<TO DO>>>


**Congratulations!** You finished this exercise by solving two problems with cuOpt LP solver.

**In the next exercise, you will work with the cuOpt Python API for solving Quadratic Programming (QP) models.**

<img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>